# Midstance Test Cases

Interactive test suite for the functions in `midstance.ipynb`.  
Run all cells top-to-bottom. A ✅ means the test passed; ❌ means it failed.

**To test a new dataset:** fill in the variables in the **New Dataset** section at the bottom and run those cells.

## 0. Imports & Notebook Test Runner

In [1]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks, savgol_filter
import traceback

# ── Lightweight test runner (no pytest needed) ──────────────────
_results = []

def run_test(name, fn):
    try:
        fn()
        _results.append((name, True, None))
        print(f'  ✅  {name}')
    except Exception as e:
        _results.append((name, False, str(e)))
        print(f'  ❌  {name}')
        print(f'       {e}')

def section(title):
    print(f'\n{'─'*55}')
    print(f'  {title}')
    print(f'{'─'*55}')

def summary():
    passed = sum(1 for _, ok, _ in _results if ok)
    total  = len(_results)
    print(f'\n{'='*55}')
    print(f'  Results: {passed}/{total} passed')
    if passed < total:
        print('  Failed:')
        for name, ok, err in _results:
            if not ok:
                print(f'    ❌ {name}: {err}')
    print(f'{'='*55}')

print('Imports ready.')

Imports ready.


## 1. Notebook Functions

Copied from `midstance.ipynb` — keep these in sync if you update the notebook.

In [2]:
# Sensor groups
HEEL_SENSORS     = ['pressure_08', 'pressure_11']
FOREFOOT_SENSORS = ['pressure_02', 'pressure_04', 'pressure_05',
                    'pressure_06', 'pressure_09']

# Step segmentation parameters
SMOOTH_WIN    = 11
SMOOTH_POLY   = 2
BIG_PEAK_DIST = 50
BIG_PEAK_PROM = 200
BIG_PEAK_HT   = 500

In [3]:
def preprocess(df):
    """Sort by timestamp and add time_sec column."""
    df = df.copy().sort_values('timestamp').reset_index(drop=True)
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['time_sec']  = (df['timestamp'] - df['timestamp'].iloc[0]) / 1000.0
    return df

In [4]:
def find_step_windows(df, sensors=None):
    """Return stance-peak indices from the summed heel signal."""
    if sensors is None:
        sensors = HEEL_SENSORS
    y_comb = sum(
        pd.to_numeric(df[s], errors='coerce').fillna(0).to_numpy(dtype=float)
        for s in sensors if s in df.columns
    )
    n  = len(y_comb)
    sw = min(SMOOTH_WIN, n - (0 if n % 2 == 1 else 1))
    if sw % 2 == 0: sw -= 1
    if sw <= SMOOTH_POLY:
        sw = SMOOTH_POLY + 3 + (1 if (SMOOTH_POLY + 3) % 2 == 0 else 0)
    y_sm = savgol_filter(y_comb, window_length=sw,
                         polyorder=min(SMOOTH_POLY, sw - 1))
    peaks, _ = find_peaks(y_sm, distance=BIG_PEAK_DIST,
                           prominence=BIG_PEAK_PROM, height=BIG_PEAK_HT)
    return peaks

In [5]:
def extract_midstance_pressure(
    df,
    surface_label='unknown',
    foot_label='unknown',
    time_col='time_sec',
    heel_sensors=None,
    forefoot_sensors=None,
    smooth_win=11,
    smooth_poly=2,
    peak_dist=50,
    peak_prom=200,
    peak_ht=500,
):
    """
    Segment steps and find midstance for each step.
    Returns (DataFrame of midstance rows, array of peak indices).
    """
    if heel_sensors     is None: heel_sensors     = HEEL_SENSORS
    if forefoot_sensors is None: forefoot_sensors = FOREFOOT_SENSORS

    df = df.copy().reset_index(drop=True)
    heel_cols = [c for c in heel_sensors     if c in df.columns]
    fore_cols = [c for c in forefoot_sensors if c in df.columns]
    n_heel, n_fore = len(heel_cols), len(fore_cols)

    heel_sig  = df[heel_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1).fillna(0).values
    fore_sig  = df[fore_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1).fillna(0).values
    heel_norm = heel_sig / max(n_heel, 1)
    fore_norm = fore_sig / max(n_fore, 1)

    n  = len(heel_sig)
    sw = min(smooth_win, n - (0 if n % 2 == 1 else 1))
    if sw % 2 == 0: sw -= 1
    if sw <= smooth_poly:
        sw = smooth_poly + 3 + (1 if (smooth_poly + 3) % 2 == 0 else 0)
    y_sm = savgol_filter(heel_sig, window_length=sw,
                         polyorder=min(smooth_poly, sw - 1))
    big_peaks, _ = find_peaks(y_sm, distance=peak_dist,
                               prominence=peak_prom, height=peak_ht)

    if len(big_peaks) < 2:
        return pd.DataFrame(), big_peaks

    results = []
    for i in range(len(big_peaks) - 1):
        p1, p2  = big_peaks[i], big_peaks[i + 1]
        c_idx   = p1 + int(np.argmin(heel_sig[p1:p2]))
        diff    = np.abs(heel_norm[c_idx:p2] - fore_norm[c_idx:p2])
        mid_idx = c_idx + int(np.argmin(diff))

        results.append({
            'surface'              : surface_label,
            'foot'                 : foot_label,
            'step_id'              : i + 1,
            'contact_time_sec'     : float(df[time_col].iloc[c_idx]),
            'midstance_time_sec'   : float(df[time_col].iloc[mid_idx]),
            'midstance_heel'       : float(heel_norm[mid_idx]),
            'midstance_forefoot'   : float(fore_norm[mid_idx]),
            'midstance_difference' : float(diff[mid_idx - c_idx]),
        })

    return pd.DataFrame(results), big_peaks

print('Functions defined.')

Functions defined.


## 2. Synthetic Data Helpers

These build fake DataFrames with known step patterns so tests don't depend on external files.

In [6]:
ALL_PRESSURE_COLS = [f'pressure_{i:02d}' for i in range(1, 13)]

def make_base_df(n_rows=500, start_ts=1_000_000, interval_ms=17, sole_id=1):
    """Return a minimal valid raw DataFrame (all pressures zero)."""
    ts = np.arange(start_ts, start_ts + n_rows * interval_ms,
                   interval_ms, dtype=int)[:n_rows]
    df = pd.DataFrame({'sole_id': sole_id, 'timestamp': ts})
    for col in ALL_PRESSURE_COLS:
        df[col] = 0
    for col in ['accel_x','accel_y','accel_z',
                'gyro_x','gyro_y','gyro_z',
                'magn_x','magn_y','magn_z','corrupt']:
        df[col] = 0
    return df

def inject_steps(df, n_steps=5, step_period=80, heel_amp=2000, fore_amp=1500):
    """Inject Gaussian-shaped heel + forefoot pulses into a base DataFrame."""
    df = df.copy()
    n  = len(df)
    for k in range(n_steps):
        centre = (k + 1) * step_period
        for i in range(n):
            heel_val = int(heel_amp * np.exp(-0.5 * ((i - centre) / 8) ** 2))
            fore_val = int(fore_amp * np.exp(-0.5 * ((i - centre - 15) / 8) ** 2))
            for col in HEEL_SENSORS:
                df.loc[i, col] += heel_val
            for col in FOREFOOT_SENSORS:
                df.loc[i, col] += fore_val
    return df

def make_preprocessed(n_steps=5, n_rows=600):
    """Convenience: build + inject + preprocess in one call."""
    df = make_base_df(n_rows=n_rows)
    df = inject_steps(df, n_steps=n_steps)
    return preprocess(df)

print('Helpers ready.')

Helpers ready.


## 3. Tests: `preprocess()`

In [7]:
section('preprocess()')

def t_returns_dataframe():
    result = preprocess(make_base_df())
    assert isinstance(result, pd.DataFrame)

def t_time_sec_starts_at_zero():
    result = preprocess(make_base_df(start_ts=9_999_000))
    assert result['time_sec'].iloc[0] == 0.0

def t_time_sec_monotonic():
    result = preprocess(make_base_df())
    assert (np.diff(result['time_sec'].values) > 0).all()

def t_sorted_even_if_shuffled():
    df = make_base_df()
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    result = preprocess(df)
    assert (np.diff(result['timestamp'].values) >= 0).all()

def t_time_sec_scale():
    result = preprocess(make_base_df(n_rows=100, interval_ms=17))
    diffs  = np.diff(result['time_sec'].values)
    assert abs(diffs.mean() - 0.017) < 1e-4

def t_bad_timestamp_coerced():
    df = make_base_df()
    df.loc[5, 'timestamp'] = 'bad'
    result = preprocess(df)   # should not raise
    assert 'time_sec' in result.columns

for name, fn in [
    ('returns DataFrame',                t_returns_dataframe),
    ('time_sec starts at zero',          t_time_sec_starts_at_zero),
    ('time_sec monotonically increasing',t_time_sec_monotonic),
    ('sorted even if input shuffled',    t_sorted_even_if_shuffled),
    ('time_sec scale (~0.017 s)',        t_time_sec_scale),
    ('bad timestamp coerced gracefully', t_bad_timestamp_coerced),
]:
    run_test(name, fn)


───────────────────────────────────────────────────────
  preprocess()
───────────────────────────────────────────────────────
  ✅  returns DataFrame
  ✅  time_sec starts at zero
  ✅  time_sec monotonically increasing
  ✅  sorted even if input shuffled
  ✅  time_sec scale (~0.017 s)
  ❌  bad timestamp coerced gracefully
       '<' not supported between instances of 'str' and 'int'


/var/folders/5_/nwk_4h3945ncxf6z1z8yzn040000gn/T/ipykernel_19112/1329124512.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'bad' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[5, 'timestamp'] = 'bad'


## 4. Tests: `find_step_windows()`

In [8]:
section('find_step_windows()')

def t_returns_ndarray():
    peaks = find_step_windows(make_preprocessed())
    assert isinstance(peaks, np.ndarray)

def t_detects_correct_count():
    df    = make_preprocessed(n_steps=5)
    peaks = find_step_windows(df)
    assert abs(len(peaks) - 5) <= 1, f'Expected ~5 peaks, got {len(peaks)}'

def t_no_steps_returns_empty():
    df    = preprocess(make_base_df())
    peaks = find_step_windows(df)
    assert len(peaks) == 0

def t_indices_in_bounds():
    df    = make_preprocessed()
    peaks = find_step_windows(df)
    assert all(0 <= p < len(df) for p in peaks)

def t_custom_sensor_list():
    df    = make_preprocessed()
    peaks = find_step_windows(df, sensors=['pressure_08'])
    assert isinstance(peaks, np.ndarray)

def t_short_df_no_crash():
    df    = preprocess(make_base_df(n_rows=20))
    peaks = find_step_windows(df)
    assert isinstance(peaks, np.ndarray)

for name, fn in [
    ('returns ndarray',                  t_returns_ndarray),
    ('detects ~correct step count',      t_detects_correct_count),
    ('no steps → empty array',           t_no_steps_returns_empty),
    ('peak indices within bounds',       t_indices_in_bounds),
    ('custom sensor list accepted',      t_custom_sensor_list),
    ('short dataframe does not crash',   t_short_df_no_crash),
]:
    run_test(name, fn)


───────────────────────────────────────────────────────
  find_step_windows()
───────────────────────────────────────────────────────
  ✅  returns ndarray
  ✅  detects ~correct step count
  ✅  no steps → empty array
  ✅  peak indices within bounds
  ✅  custom sensor list accepted
  ✅  short dataframe does not crash


## 5. Tests: `extract_midstance_pressure()`

In [9]:
section('extract_midstance_pressure()')

def t_returns_tuple():
    result = extract_midstance_pressure(make_preprocessed())
    assert isinstance(result, tuple) and len(result) == 2

def t_output_columns():
    mid_df, _ = extract_midstance_pressure(
        make_preprocessed(), surface_label='test', foot_label='left')
    required = {'surface','foot','step_id','contact_time_sec',
                'midstance_time_sec','midstance_heel',
                'midstance_forefoot','midstance_difference'}
    assert required.issubset(set(mid_df.columns))

def t_step_count_reasonable():
    n_steps   = 5
    mid_df, _ = extract_midstance_pressure(make_preprocessed(n_steps=n_steps))
    assert len(mid_df) >= n_steps - 2

def t_labels_propagated():
    mid_df, _ = extract_midstance_pressure(
        make_preprocessed(), surface_label='concrete', foot_label='right')
    assert (mid_df['surface'] == 'concrete').all()
    assert (mid_df['foot']    == 'right').all()

def t_step_ids_sequential():
    mid_df, _ = extract_midstance_pressure(make_preprocessed())
    assert list(mid_df['step_id']) == list(range(1, len(mid_df) + 1))

def t_midstance_after_contact():
    mid_df, _ = extract_midstance_pressure(make_preprocessed())
    assert (mid_df['midstance_time_sec'] >= mid_df['contact_time_sec']).all()

def t_difference_non_negative():
    mid_df, _ = extract_midstance_pressure(make_preprocessed())
    assert (mid_df['midstance_difference'] >= 0).all()

def t_pressure_non_negative():
    mid_df, _ = extract_midstance_pressure(make_preprocessed())
    assert (mid_df['midstance_heel']     >= 0).all()
    assert (mid_df['midstance_forefoot'] >= 0).all()

def t_contact_times_increasing():
    mid_df, _ = extract_midstance_pressure(make_preprocessed(n_steps=6))
    assert (np.diff(mid_df['contact_time_sec'].values) > 0).all()

def t_no_peaks_returns_empty():
    mid_df, _ = extract_midstance_pressure(preprocess(make_base_df()))
    assert isinstance(mid_df, pd.DataFrame) and len(mid_df) == 0

def t_missing_forefoot_cols_no_crash():
    df = make_preprocessed()
    df = df.drop(columns=['pressure_02', 'pressure_04'])
    mid_df, _ = extract_midstance_pressure(df)
    assert isinstance(mid_df, pd.DataFrame)

def t_low_peak_ht_detects_more():
    df           = make_preprocessed(n_steps=4)
    mid_default, _ = extract_midstance_pressure(df, peak_ht=500)
    mid_low,     _ = extract_midstance_pressure(df, peak_ht=100)
    assert len(mid_low) >= len(mid_default)

for name, fn in [
    ('returns (DataFrame, peaks) tuple',      t_returns_tuple),
    ('output has required columns',           t_output_columns),
    ('step count is reasonable',              t_step_count_reasonable),
    ('surface & foot labels propagated',      t_labels_propagated),
    ('step_id values are sequential',         t_step_ids_sequential),
    ('midstance_time >= contact_time',        t_midstance_after_contact),
    ('midstance_difference non-negative',     t_difference_non_negative),
    ('pressure values non-negative',          t_pressure_non_negative),
    ('contact times monotonically increasing',t_contact_times_increasing),
    ('no peaks → empty DataFrame returned',   t_no_peaks_returns_empty),
    ('missing forefoot cols handled',         t_missing_forefoot_cols_no_crash),
    ('lower peak_ht detects ≥ more steps',    t_low_peak_ht_detects_more),
]:
    run_test(name, fn)


───────────────────────────────────────────────────────
  extract_midstance_pressure()
───────────────────────────────────────────────────────
  ✅  returns (DataFrame, peaks) tuple
  ✅  output has required columns
  ✅  step count is reasonable
  ✅  surface & foot labels propagated
  ✅  step_id values are sequential
  ✅  midstance_time >= contact_time
  ✅  midstance_difference non-negative
  ✅  pressure values non-negative
  ✅  contact times monotonically increasing
  ✅  no peaks → empty DataFrame returned
  ✅  missing forefoot cols handled
  ✅  lower peak_ht detects ≥ more steps


## 6. Unit Test Summary

In [10]:
summary()


  Results: 23/24 passed
  Failed:
    ❌ bad timestamp coerced gracefully: '<' not supported between instances of 'str' and 'int'


---
## 7. Predict Steps on a New Dataset

Fill in the variables below and run the remaining cells.  
The pipeline will load your CSV, detect steps, find midstance for each step, and print a summary table.

In [21]:
# ── Configuration — edit these ─────────────────────────────────
NEW_CSV_PATH       = 'Kristian_dry_0425.csv'   # path to your CSV file
SURFACE_LABEL      = 'new_surface'     
FOOT_LABEL         = 'left'                 
SOLE_ID            = 2                     
EXPECTED_MIN_STEPS = 5                     
EXPECTED_MAX_STEPS = 350
# ──────────────────────────────────────────────────────────────

In [22]:
import os

if not os.path.exists(NEW_CSV_PATH):
    print(f'⚠️  File not found: {NEW_CSV_PATH}')
    print('   Update NEW_CSV_PATH above and re-run.')
else:
    raw_new = pd.read_csv(NEW_CSV_PATH, low_memory=False)
    df_new  = preprocess(raw_new[raw_new['sole_id'] == SOLE_ID])
    print(f'Loaded  : {NEW_CSV_PATH}')
    print(f'Rows    : {len(df_new)}')
    print(f'Duration: {df_new["time_sec"].iloc[-1]:.1f} s')
    print(f'Columns : {list(df_new.columns)}')

Loaded  : Kristian_dry_0425.csv
Rows    : 25634
Duration: 410.1 s
Columns : ['sole_id', 'timestamp', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 'magn_x', 'magn_y', 'magn_z', 'pressure_01', 'pressure_02', 'pressure_03', 'pressure_04', 'pressure_05', 'pressure_06', 'pressure_07', 'pressure_08', 'pressure_09', 'pressure_10', 'pressure_11', 'pressure_12', 'corrupt', 'time_sec']


In [23]:
if 'df_new' in dir() and len(df_new) > 0:
    mid_new, peaks_new = extract_midstance_pressure(
        df_new,
        surface_label = SURFACE_LABEL,
        foot_label    = FOOT_LABEL,
    )
    print(f'Steps detected: {len(mid_new)}')
    mid_new

Steps detected: 347


In [24]:
if 'mid_new' in dir() and len(mid_new) > 0:
    section(f'Integration tests — {NEW_CSV_PATH}')
    _results.clear()   # fresh slate for integration results

    def ti_step_count():
        n = len(mid_new)
        assert EXPECTED_MIN_STEPS <= n <= EXPECTED_MAX_STEPS, (
            f'Got {n} steps; expected [{EXPECTED_MIN_STEPS}, {EXPECTED_MAX_STEPS}]')

    def ti_required_columns():
        required = {'surface','foot','step_id','contact_time_sec',
                    'midstance_time_sec','midstance_heel',
                    'midstance_forefoot','midstance_difference'}
        assert required.issubset(set(mid_new.columns))

    def ti_midstance_after_contact():
        assert (mid_new['midstance_time_sec'] >= mid_new['contact_time_sec']).all()

    def ti_no_nans():
        assert not mid_new.isnull().any().any(), 'NaNs found in output'

    def ti_pressures_non_negative():
        assert (mid_new['midstance_heel']     >= 0).all()
        assert (mid_new['midstance_forefoot'] >= 0).all()

    def ti_step_ids_sequential():
        assert list(mid_new['step_id']) == list(range(1, len(mid_new) + 1))

    def ti_labels_correct():
        assert (mid_new['surface'] == SURFACE_LABEL).all()
        assert (mid_new['foot']    == FOOT_LABEL).all()

    for name, fn in [
        (f'step count in [{EXPECTED_MIN_STEPS}, {EXPECTED_MAX_STEPS}]', ti_step_count),
        ('all required columns present',      ti_required_columns),
        ('midstance_time >= contact_time',    ti_midstance_after_contact),
        ('no NaNs in output',                 ti_no_nans),
        ('pressure values non-negative',      ti_pressures_non_negative),
        ('step_ids are sequential',           ti_step_ids_sequential),
        ('surface & foot labels correct',     ti_labels_correct),
    ]:
        run_test(name, fn)

    summary()


───────────────────────────────────────────────────────
  Integration tests — Kristian_dry_0425.csv
───────────────────────────────────────────────────────
  ✅  step count in [5, 350]
  ✅  all required columns present
  ✅  midstance_time >= contact_time
  ✅  no NaNs in output
  ✅  pressure values non-negative
  ✅  step_ids are sequential
  ✅  surface & foot labels correct

  Results: 7/7 passed


In [25]:
if 'mid_new' in dir() and len(mid_new) > 0:
    print('\n── Summary statistics ──────────────────────────────────')
    print(f'  Surface : {SURFACE_LABEL}   Foot: {FOOT_LABEL}')
    print(f'  Steps   : {len(mid_new)}')
    print(f'  Mean midstance heel     : {mid_new["midstance_heel"].mean():.1f}')
    print(f'  Mean midstance forefoot : {mid_new["midstance_forefoot"].mean():.1f}')
    print(f'  Mean difference         : {mid_new["midstance_difference"].mean():.1f}')
    print(f'  Contact time range      : {mid_new["contact_time_sec"].min():.2f} – {mid_new["contact_time_sec"].max():.2f} s')
    print('───────────────────────────────────────────────────────')


── Summary statistics ──────────────────────────────────
  Surface : new_surface   Foot: left
  Steps   : 347
  Mean midstance heel     : 320.1
  Mean midstance forefoot : 318.2
  Mean difference         : 16.3
  Contact time range      : 2.85 – 408.80 s
───────────────────────────────────────────────────────
